In [ ]:
import plotly.figure_factory as ff
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from jaxtyping import Float
from plotly.subplots import make_subplots
from torch import Tensor
from tqdm.autonotebook import tqdm

from src.config.base import BaseConfig
from src.model.mlp import StackedResidualMLPConfig
from src.model.time_conditioning import TimeConditioningConfig

device = "cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
class BimodalDataConfig(BaseConfig):
    offset: Float[Tensor, "2"]
    center: float
    std: float
    scale: float
    std_cap: float = 2

    def _sample_capped_noise(self, batch_size: int) -> Float[Tensor, "batch 2"]:
        z = torch.randn((batch_size, 2))
        norms = z.norm(dim=1, keepdim=True).clamp_min(1e-6)
        capped_norms = norms.clamp_max(self.std_cap)
        return z * (self.std * capped_norms / norms)

    def sample_mode(
        self,
        batch_size: int,
        mode_id: int,
    ) -> Float[Tensor, "batch 2"]:
        x = self._sample_capped_noise(batch_size)
        direction = 1.0 if mode_id == 0 else -1.0
        x[:, 0] += direction * self.center / self.scale
        return x * self.scale + self.offset

    def sample_with_mode_ids(
        self,
        batch_size: int,
    ) -> tuple[Float[Tensor, "batch 2"], Tensor]:
        assert batch_size % 2 == 0
        per_mode_size = batch_size // 2
        mode_ids = torch.cat(
            [
                torch.zeros(per_mode_size, dtype=torch.long),
                torch.ones(per_mode_size, dtype=torch.long),
            ]
        )
        x = torch.cat(
            [
                self.sample_mode(per_mode_size, mode_id=0),
                self.sample_mode(per_mode_size, mode_id=1),
            ],
            dim=0,
        )
        return x, mode_ids

    def sample(self, batch_size: int) -> Float[Tensor, "batch 2"]:
        x, _ = self.sample_with_mode_ids(batch_size)
        return x

    def sample_prior(self, batch_size: int) -> Float[Tensor, "batch 2"]:
        return torch.randn((batch_size, 2))

    def sample_contours(
        self,
        per_contour_size: int,
        zscores: tuple[float, ...] = (0.1, 0.5, 1.0, 1.5),
    ) -> tuple[Float[Tensor, "batch 2"], Tensor, tuple[float, ...]]:
        angles = torch.linspace(0.0, 2.0 * torch.pi, per_contour_size + 1)[:-1]
        unit_circle = torch.stack([torch.cos(angles), torch.sin(angles)], dim=-1)
        contour_points = []
        contour_ids = []
        for contour_id, zscore in enumerate(zscores):
            contour_points.append(zscore * unit_circle)
            contour_ids.append(
                torch.full((per_contour_size,), contour_id, dtype=torch.long)
            )
        return torch.cat(contour_points, dim=0), torch.cat(contour_ids, dim=0), zscores


data_config = BimodalDataConfig(
    offset=torch.tensor([0.0, 0.0]),
    center=3.0,
    std=0.2,
    scale=2.0,
)
# nu, nu_mode_ids = data_config.sample_with_mode_ids(4000)
# preview = px.scatter(
#     x=nu[:, 0],
#     y=nu[:, 1],
#     color=nu_mode_ids,
# )
# preview.update_traces(marker={'size': 4, 'opacity': 0.35})
# preview.show()

# contour_points, contour_ids, contour_levels = data_config.sample_contours(
#     per_contour_size=300,
# )
# fig = go.Figure()
# for contour_id, contour_level in enumerate(contour_levels):
#     mask = contour_ids == contour_id
#     fig.add_trace(go.Scatter(
#         x=contour_points[mask, 0].cpu().numpy(),
#         y=contour_points[mask, 1].cpu().numpy(),
#         mode='lines',
#         name=f'z = {contour_level}',
#     ))
# fig.update_yaxes(scaleanchor='x')
# fig.show()


In [ ]:
class LatentDriftingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = StackedResidualMLPConfig.initialize(
            layer_dims=[2, 512, 512, 512, 512, 2],
        ).get_model()

        self.decoder = StackedResidualMLPConfig.initialize(
            layer_dims=[2, 512, 512, 512, 512, 2],
        ).get_model()

        self.noise_critic = StackedResidualMLPConfig.initialize(
            layer_dims=[2, 512, 512, 512, 512, 2],
            time_conditioning_config=TimeConditioningConfig(
                min_t_lambda=5e-3,
                max_t_lambda=1.0,
                sinusoidal_dim=512,
                hidden_dim=512,
                output_dim=512,
            ),
        ).get_model()

    def implied_denoiser(
        self,
        y_t: Float[Tensor, "batch 2"],
        t: Float[Tensor, "batch 1"],
    ) -> Float[Tensor, "batch 2"]:
        predicted_noise = self.noise_critic(y_t, t=t)
        return implied_denoiser_from_noise_prediction(
            y_t=y_t,
            predicted_noise=predicted_noise,
            t=t,
        )

    def get_optimizers(
        self,
    ) -> tuple[
        torch.optim.Optimizer,
        torch.optim.Optimizer,
        torch.optim.Optimizer,
    ]:
        return (
            torch.optim.AdamW(self.encoder.parameters(), lr=3e-4, weight_decay=1e-4),
            torch.optim.AdamW(self.decoder.parameters(), lr=3e-4, weight_decay=1e-4),
            torch.optim.AdamW(self.noise_critic.parameters(), lr=3e-4, weight_decay=1e-4),
        )


model = LatentDriftingModel().to(device)
encoder_op, decoder_op, critic_op = model.get_optimizers()
batch_size = 2048
grad_clip_norm = 1.0

logs = {
    "losses": {
        "data_cycle": [],
        "prior_cycle": [],
        "critic_denoising": [],
        "encoder_chart_drift": [],
        "decoder_chart_drift": [],
    },
    "latent_metrics": {"mean_signal": [], "cov_signal": []},
}

In [ ]:
MODE_COLORS = ("#1f77b4", "#ff7f0e")
CONTOUR_COLORS = ("#440154", "#3b528b", "#21918c", "#5ec962", "#fde725")
LOSS_COLORS = {
    "data_cycle": "#1f77b4",
    "prior_cycle": "#ff7f0e",
    "critic_denoising": "#d62728",
    "encoder_chart_drift": "#2ca02c",
    "decoder_chart_drift": "#e377c2",
}
LATENT_METRIC_COLORS = {
    "mean_signal": "#9467bd",
    "cov_signal": "#8c564b",
}


def current_training_step() -> int:
    return len(logs["losses"]["critic_denoising"])


def trailing_window(values: list[float], window_size: int = 2000) -> tuple[list[int], list[float]]:
    start = max(0, len(values) - window_size)
    return list(range(start + 1, len(values) + 1)), values[start:]


def plot_training_losses() -> go.Figure:
    step = current_training_step()
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            "training losses",
            "latent Gaussianity diagnostics",
        ],
        horizontal_spacing=0.08,
    )
    for loss_name, loss_values in logs["losses"].items():
        xs, ys = trailing_window(loss_values)
        fig.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode="lines",
                line={"width": 3, "color": LOSS_COLORS[loss_name]},
                name=loss_name,
            ),
            row=1,
            col=1,
        )
    for metric_name, metric_values in logs["latent_metrics"].items():
        xs, ys = trailing_window(metric_values)
        fig.add_trace(
            go.Scatter(
                x=xs,
                y=ys,
                mode="lines",
                line={"width": 3, "color": LATENT_METRIC_COLORS[metric_name]},
                name=metric_name,
            ),
            row=1,
            col=2,
        )
    fig.update_layout(
        title=f"training losses at step {step}",
        height=360,
        width=1450,
        margin={"l": 50, "r": 20, "t": 70, "b": 45},
        legend={"x": 1.02, "xanchor": "left", "y": 1.0, "yanchor": "top"},
    )
    fig.update_xaxes(
        title_text="optimization step",
        showgrid=True,
        zeroline=False,
        row=1,
        col=1,
    )
    fig.update_yaxes(title_text="loss", showgrid=True, zeroline=False, row=1, col=1)
    fig.update_xaxes(
        title_text="optimization step",
        showgrid=True,
        zeroline=False,
        row=1,
        col=2,
    )
    fig.update_yaxes(
        title_text="signal",
        showgrid=True,
        zeroline=False,
        row=1,
        col=2,
    )
    return fig


def plot_model_contours_and_reconstructions(
    scatter_num_samples: int,
    contour_points_per_level: int = 128,
) -> go.Figure:
    step = current_training_step()
    model_was_training = model.training
    model.eval()
    with torch.no_grad():
        x_data, mode_ids = data_config.sample_with_mode_ids(scatter_num_samples)
        x_data = x_data.to(device)
        y_data = model.encoder(x_data)
        x_recon = model.decoder(y_data)
        z_random = data_config.sample_prior(scatter_num_samples).to(device)
        x_model = model.decoder(z_random)
        z_contours, contour_ids, contour_levels = data_config.sample_contours(
            per_contour_size=contour_points_per_level,
        )
        x_contours = model.decoder(z_contours.to(device))
    if model_was_training:
        model.train()

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=[
            "data and reconstructions",
            "prior contours pushed through decoder",
            "free-form decoder samples",
        ],
        horizontal_spacing=0.06,
    )
    fig.add_trace(
        go.Scatter(
            x=x_data[:, 0].detach().cpu().numpy(),
            y=x_data[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": "#202020", "opacity": 0.18},
            name="data",
        ),
        row=1,
        col=1,
    )
    for mode_id, color in enumerate(MODE_COLORS):
        mode_mask = mode_ids == mode_id
        fig.add_trace(
            go.Scatter(
                x=x_recon[mode_mask, 0].detach().cpu().numpy(),
                y=x_recon[mode_mask, 1].detach().cpu().numpy(),
                mode="markers",
                marker={"size": 5, "color": color, "opacity": 0.6},
                name=f"reconstruction mode {mode_id}",
            ),
            row=1,
            col=1,
        )
    for contour_id, contour_level in enumerate(contour_levels):
        mask = contour_ids == contour_id
        fig.add_trace(
            go.Scatter(
                x=x_contours[mask, 0].detach().cpu().numpy(),
                y=x_contours[mask, 1].detach().cpu().numpy(),
                mode="lines",
                line={
                    "width": 2.5,
                    "color": CONTOUR_COLORS[contour_id % len(CONTOUR_COLORS)],
                },
                name=f"z = {contour_level}",
            ),
            row=1,
            col=2,
        )
    fig.add_trace(
        go.Scatter(
            x=x_model[:, 0].detach().cpu().numpy(),
            y=x_model[:, 1].detach().cpu().numpy(),
            mode="markers",
            marker={"size": 4, "color": "#111111", "opacity": 0.28},
            name="decoder(prior)",
        ),
        row=1,
        col=3,
    )
    for col in [1, 2, 3]:
        fig.update_xaxes(
            showgrid=True,
            zeroline=False,
            scaleanchor=f"y{col if col > 1 else ''}",
            scaleratio=1,
            row=1,
            col=col,
        )
        fig.update_yaxes(
            showgrid=True,
            zeroline=False,
            row=1,
            col=col,
        )
    fig.update_layout(
        title=f"model snapshot at step {step}",
        height=430,
        width=1300,
        margin={"l": 30, "r": 20, "t": 85, "b": 30},
        legend={"x": 1.02, "xanchor": "left", "y": 1.0, "yanchor": "top"},
    )
    return fig


In [ ]:
def implied_denoiser_from_noise_prediction(
    y_t: Float[Tensor, "batch 2"],
    predicted_noise: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    return (y_t - t * predicted_noise) / (1.0 - t).clamp_min(1e-4)


def analytic_gaussian_implied_denoiser(
    y_t: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    variance = (1.0 - t).square() + t.square()
    return ((1.0 - t) / variance) * y_t


def score_from_noise_prediction(
    predicted_noise: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    return -predicted_noise / t.clamp_min(1e-4)


def score_from_implied_denoiser(
    y_t: Float[Tensor, "batch 2"],
    implied_denoiser: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    return ((1.0 - t) * implied_denoiser - y_t) / t.square().clamp_min(1e-6)


def evaluate_model_score(
    y_t: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    predicted_noise = model.noise_critic(y_t, t=t)
    return score_from_noise_prediction(predicted_noise, t=t)


def evaluate_gaussian_score(
    y_t: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    analytic_denoiser = analytic_gaussian_implied_denoiser(y_t, t=t)
    return score_from_implied_denoiser(
        y_t=y_t,
        implied_denoiser=analytic_denoiser,
        t=t,
    )


def evaluate_score_difference(
    y_t: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    return evaluate_model_score(y_t, t=t) - evaluate_gaussian_score(y_t, t=t)


def subsample_indices(total_size: int, max_points: int) -> Tensor:
    if total_size <= max_points:
        return torch.arange(total_size)
    return torch.linspace(0, total_size - 1, max_points).round().long()


def normalize_vectors(
    vectors: Float[Tensor, "batch 2"],
    display_length: float,
) -> tuple[Float[Tensor, "batch 2"], Float[Tensor, "batch"]]:
    magnitudes = vectors.norm(dim=1)
    normalized = vectors / magnitudes.unsqueeze(-1).clamp_min(1e-6)
    return display_length * normalized, magnitudes


def regular_grid_display_length(
    xs: Tensor,
    ys: Tensor,
    *,
    fraction: float = 1.15,
) -> float:
    if xs.numel() < 2 or ys.numel() < 2:
        return fraction
    x_step = float((xs[1] - xs[0]).item())
    y_step = float((ys[1] - ys[0]).item())
    return fraction * min(abs(x_step), abs(y_step))


def square_grid_limits(
    points: Float[Tensor, "batch 2"],
    padding: float = 0.12,
) -> tuple[float, float, float, float]:
    mins = points.min(dim=0).values
    maxs = points.max(dim=0).values
    center = 0.5 * (mins + maxs)
    radius = 0.5 * float((maxs - mins).max().item())
    radius = max(radius * (1.0 + padding), 1.0)
    return (
        float(center[0] - radius),
        float(center[0] + radius),
        float(center[1] - radius),
        float(center[1] + radius),
    )


def build_square_grid(
    points: Float[Tensor, "batch 2"],
    resolution: int,
) -> tuple[Float[Tensor, "grid 2"], Tensor, Tensor]:
    x_min, x_max, y_min, y_max = square_grid_limits(points)
    xs = torch.linspace(
        x_min,
        x_max,
        resolution,
        device=points.device,
        dtype=points.dtype,
    )
    ys = torch.linspace(
        y_min,
        y_max,
        resolution,
        device=points.device,
        dtype=points.dtype,
    )
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
    grid_points = torch.stack([grid_x.reshape(-1), grid_y.reshape(-1)], dim=-1)
    return grid_points, xs.detach().cpu(), ys.detach().cpu()


def add_quiver_traces(
    fig: go.Figure,
    *,
    points: Float[Tensor, "batch 2"],
    vectors: Float[Tensor, "batch 2"],
    magnitudes: Float[Tensor, "batch"],
    color: str,
    name: str,
    row: int,
    col: int,
    visible: bool,
    showlegend: bool,
    show_hover: bool = True,
    arrow_scale: float = 0.32,
    line_width: float = 1.2,
    opacity: float = 0.65,
) -> None:
    quiver = ff.create_quiver(
        x=points[:, 0].detach().cpu().float().tolist(),
        y=points[:, 1].detach().cpu().float().tolist(),
        u=vectors[:, 0].detach().cpu().float().tolist(),
        v=vectors[:, 1].detach().cpu().float().tolist(),
        scale=1.0,
        arrow_scale=arrow_scale,
        line_color=color,
        name=name,
    )
    for trace_id, trace in enumerate(quiver.data):
        trace.visible = visible
        trace.showlegend = showlegend and trace_id == 0
        trace.name = name
        trace.hoverinfo = "skip"
        trace.line.width = line_width
        trace.opacity = opacity
        fig.add_trace(trace, row=row, col=col)

    if not show_hover:
        return

    hover_text = [
        f"{name}<br>|v|={float(magnitude):.4f}"
        for magnitude in magnitudes.detach().cpu().float().tolist()
    ]
    fig.add_trace(
        go.Scatter(
            x=points[:, 0].detach().cpu().float().tolist(),
            y=points[:, 1].detach().cpu().float().tolist(),
            mode="markers",
            marker={"size": 10, "color": color, "opacity": 0.001},
            text=hover_text,
            hovertemplate="%{text}<extra></extra>",
            showlegend=False,
            name=name,
            visible=visible,
        ),
        row=row,
        col=col,
    )


def add_model_and_gaussian_score_hover_trace(
    fig: go.Figure,
    *,
    points: Float[Tensor, "batch 2"],
    model_score_magnitudes: Float[Tensor, "batch"],
    gaussian_score_magnitudes: Float[Tensor, "batch"],
    row: int,
    col: int,
    visible: bool,
) -> None:
    hover_text = [
        (
            f"model score field<br>|v_model|={float(model_magnitude):.4f}<br>"
            f"analytic Gaussian score field<br>|v_gaussian|={float(gaussian_magnitude):.4f}"
        )
        for model_magnitude, gaussian_magnitude in zip(
            model_score_magnitudes.detach().cpu().float().tolist(),
            gaussian_score_magnitudes.detach().cpu().float().tolist(),
            strict=True,
        )
    ]
    fig.add_trace(
        go.Scatter(
            x=points[:, 0].detach().cpu().float().tolist(),
            y=points[:, 1].detach().cpu().float().tolist(),
            mode="markers",
            marker={"size": 12, "color": "#111111", "opacity": 0.001},
            text=hover_text,
            hovertemplate="%{text}<extra></extra>",
            showlegend=False,
            name="model / analytic Gaussian score magnitudes",
            visible=visible,
        ),
        row=row,
        col=col,
    )


def add_latent_contour_traces(
    fig: go.Figure,
    *,
    contour_points: Float[Tensor, "batch 2"],
    contour_ids: Tensor,
    contour_levels: tuple[float, ...],
    row: int,
    col: int,
    visible: bool = True,
    showlegend: bool = False,
) -> None:
    for contour_id, contour_level in enumerate(contour_levels):
        mask = contour_ids == contour_id
        fig.add_trace(
            go.Scatter(
                x=contour_points[mask, 0].detach().cpu().numpy(),
                y=contour_points[mask, 1].detach().cpu().numpy(),
                mode="lines",
                line={
                    "width": 2.0,
                    "color": CONTOUR_COLORS[contour_id % len(CONTOUR_COLORS)],
                    "dash": "dot",
                },
                name=f"latent z = {contour_level}",
                visible=visible,
                showlegend=showlegend,
            ),
            row=row,
            col=col,
        )


def add_mismatch_contour_trace(
    fig: go.Figure,
    *,
    xs: Tensor,
    ys: Tensor,
    mismatch_norm: Float[Tensor, "y x"],
    row: int,
    col: int,
    visible: bool,
    name: str,
) -> None:
    fig.add_trace(
        go.Contour(
            x=xs.numpy(),
            y=ys.numpy(),
            z=mismatch_norm.detach().cpu().numpy(),
            colorscale="Viridis",
            ncontours=18,
            contours={"coloring": "heatmap", "showlabels": False},
            line={"width": 0.8},
            opacity=0.38,
            showscale=False,
            hovertemplate="x=%{x:.2f}<br>y=%{y:.2f}<br>||Δscore||=%{z:.4f}<extra></extra>",
            name=name,
            visible=visible,
        ),
        row=row,
        col=col,
    )


def estimate_noise_prediction_loss(
    clean_latents: Float[Tensor, "batch 2"],
    noise_level: float,
    num_noise_draws: int,
) -> float:
    t = torch.full(
        (clean_latents.shape[0], 1),
        float(noise_level),
        device=clean_latents.device,
        dtype=clean_latents.dtype,
    )
    total_loss = clean_latents.new_tensor(0.0)
    with torch.no_grad():
        for _ in range(num_noise_draws):
            noise = torch.randn_like(clean_latents)
            noisy_latents = (1.0 - t) * clean_latents + t * noise
            predicted_noise = model.noise_critic(noisy_latents, t=t)
            total_loss = total_loss + F.mse_loss(predicted_noise, noise)
    return float((total_loss / num_noise_draws).item())


def implied_denoiser_mismatch_gradients(
    clean_latents: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
    noise: Float[Tensor, "batch 2"],
) -> tuple[
    Float[Tensor, "batch 2"],
    Float[Tensor, "batch 2"],
    Float[Tensor, "batch 2"],
]:
    noisy_latents = ((1.0 - t) * clean_latents + t * noise).detach().clone()
    noisy_latents.requires_grad_(True)
    predicted_noise = model.noise_critic(noisy_latents, t=t)
    implied_denoiser = implied_denoiser_from_noise_prediction(
        y_t=noisy_latents,
        predicted_noise=predicted_noise,
        t=t,
    )
    analytic_denoiser = analytic_gaussian_implied_denoiser(y_t=noisy_latents, t=t)
    mismatch = (implied_denoiser - analytic_denoiser).square().mean(dim=1)
    noisy_latent_drift = -torch.autograd.grad(mismatch.sum(), noisy_latents)[0]
    clean_latent_drift = (1.0 - t) * noisy_latent_drift
    return (
        noisy_latents.detach(),
        noisy_latent_drift.detach(),
        clean_latent_drift.detach(),
    )


def score_difference_transport_vectors(
    clean_latents: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
    noise: Float[Tensor, "batch 2"],
) -> tuple[
    Float[Tensor, "batch 2"],
    Float[Tensor, "batch 2"],
    Float[Tensor, "batch 2"],
]:
    noisy_latents = ((1.0 - t) * clean_latents + t * noise).detach()
    noisy_transport = -evaluate_score_difference(noisy_latents, t=t)
    clean_transport = (1.0 - t) * noisy_transport
    return (
        noisy_latents.detach(),
        noisy_transport.detach(),
        clean_transport.detach(),
    )


def add_potential_contour_trace(
    fig: go.Figure,
    *,
    xs: Tensor,
    ys: Tensor,
    potential: Float[Tensor, "y x"],
    row: int,
    col: int,
    visible: bool,
    name: str,
) -> None:
    fig.add_trace(
        go.Contour(
            x=xs.numpy(),
            y=ys.numpy(),
            z=potential.detach().cpu().numpy(),
            colorscale="RdBu",
            ncontours=18,
            contours={"coloring": "heatmap", "showlabels": False},
            line={"width": 0.8},
            opacity=0.38,
            showscale=False,
            hovertemplate="x=%{x:.2f}<br>y=%{y:.2f}<br>potential=%{z:.4f}<extra></extra>",
            name=name,
            visible=visible,
        ),
        row=row,
        col=col,
    )


def plot_score_estimates(
    scatter_num_samples: int,
    plot_t_values: list[float],
    vector_num_samples: int = 64,
    loss_num_noise_draws: int = 8,
    contour_points_per_level: int = 128,
    contour_grid_resolution: int = 144,
    vector_grid_resolution: int = 34,
) -> go.Figure:
    step = current_training_step()
    model_was_training = model.training
    model.eval()
    x_data, mode_ids = data_config.sample_with_mode_ids(scatter_num_samples)
    x_data = x_data.to(device)
    mode_ids = mode_ids.to(device)
    contour_points, contour_ids, contour_levels = data_config.sample_contours(
        per_contour_size=contour_points_per_level,
    )
    with torch.no_grad():
        clean_latents = model.encoder(x_data)

    snapshots = []
    loss_estimates = []
    for noise_level in plot_t_values:
        t = torch.full(
            (clean_latents.shape[0], 1),
            float(noise_level),
            device=clean_latents.device,
            dtype=clean_latents.dtype,
        )
        noise = torch.randn_like(clean_latents)
        with torch.no_grad():
            noisy_latents = (1.0 - t) * clean_latents + t * noise
            predicted_noise = model.noise_critic(noisy_latents, t=t)
            score_vectors = score_from_noise_prediction(predicted_noise, t=t)
            gaussian_score_vectors = evaluate_gaussian_score(noisy_latents, t=t)

            noisy_grid_points, noisy_xs, noisy_ys = build_square_grid(
                noisy_latents,
                resolution=contour_grid_resolution,
            )
            noisy_vector_display_length = regular_grid_display_length(
                noisy_xs,
                noisy_ys,
            )
            noisy_grid_t = torch.full(
                (noisy_grid_points.shape[0], 1),
                float(noise_level),
                device=clean_latents.device,
                dtype=clean_latents.dtype,
            )
            noisy_score_difference = evaluate_score_difference(
                noisy_grid_points,
                t=noisy_grid_t,
            )
            noisy_mismatch_norm = noisy_score_difference.norm(dim=1).reshape(
                contour_grid_resolution,
                contour_grid_resolution,
            )

            clean_grid_points, clean_xs, clean_ys = build_square_grid(
                clean_latents,
                resolution=contour_grid_resolution,
            )
            clean_vector_display_length = regular_grid_display_length(
                clean_xs,
                clean_ys,
            )
            clean_grid_t = torch.full(
                (clean_grid_points.shape[0], 1),
                float(noise_level),
                device=clean_latents.device,
                dtype=clean_latents.dtype,
            )
            pulled_noisy_grid_points = (1.0 - clean_grid_t) * clean_grid_points
            clean_pullback_score_difference = (
                1.0 - clean_grid_t
            ) * evaluate_score_difference(
                pulled_noisy_grid_points,
                t=clean_grid_t,
            )
            clean_pullback_mismatch_norm = clean_pullback_score_difference.norm(
                dim=1
            ).reshape(
                contour_grid_resolution,
                contour_grid_resolution,
            )
            clean_transport_potential = (
                clean_grid_points * clean_pullback_score_difference.detach()
            ).sum(dim=1).reshape(
                contour_grid_resolution,
                contour_grid_resolution,
            )

            noisy_vector_grid_points, _, _ = build_square_grid(
                noisy_latents,
                resolution=vector_grid_resolution,
            )
            noisy_vector_t = torch.full(
                (noisy_vector_grid_points.shape[0], 1),
                float(noise_level),
                device=clean_latents.device,
                dtype=clean_latents.dtype,
            )
            noisy_grid_model_score = evaluate_model_score(
                noisy_vector_grid_points,
                t=noisy_vector_t,
            )
            noisy_grid_gaussian_score = evaluate_gaussian_score(
                noisy_vector_grid_points,
                t=noisy_vector_t,
            )

            clean_vector_grid_points, _, _ = build_square_grid(
                clean_latents,
                resolution=vector_grid_resolution,
            )
            clean_vector_t = torch.full(
                (clean_vector_grid_points.shape[0], 1),
                float(noise_level),
                device=clean_latents.device,
                dtype=clean_latents.dtype,
            )
            clean_grid_score_difference = (
                1.0 - clean_vector_t
            ) * evaluate_score_difference(
                (1.0 - clean_vector_t) * clean_vector_grid_points,
                t=clean_vector_t,
            )

        noisy_drift_points, noisy_drift_vectors, clean_drift_vectors = (
            implied_denoiser_mismatch_gradients(
                clean_latents=clean_latents,
                t=t,
                noise=noise,
            )
        )
        _, _, clean_transport_vectors = score_difference_transport_vectors(
            clean_latents=clean_latents,
            t=t,
            noise=noise,
        )
        loss_estimates.append(
            estimate_noise_prediction_loss(
                clean_latents=clean_latents,
                noise_level=noise_level,
                num_noise_draws=loss_num_noise_draws,
            )
        )
        score_indices = subsample_indices(
            noisy_latents.shape[0], vector_num_samples
        ).to(device)
        noisy_drift_indices = subsample_indices(
            noisy_drift_points.shape[0], vector_num_samples
        ).to(device)
        clean_drift_indices = subsample_indices(
            clean_latents.shape[0], vector_num_samples
        ).to(device)
        clean_transport_indices = subsample_indices(
            clean_latents.shape[0], vector_num_samples
        ).to(device)
        score_display_vectors, score_magnitudes = normalize_vectors(
            score_vectors[score_indices].detach().cpu(),
            0.32,
        )
        gaussian_score_display_vectors, gaussian_score_magnitudes = normalize_vectors(
            gaussian_score_vectors[score_indices].detach().cpu(),
            0.32,
        )
        noisy_drift_display_vectors, noisy_drift_magnitudes = normalize_vectors(
            noisy_drift_vectors[noisy_drift_indices].detach().cpu(),
            0.24,
        )
        clean_drift_display_vectors, clean_drift_magnitudes = normalize_vectors(
            clean_drift_vectors[clean_drift_indices].detach().cpu(),
            0.22,
        )
        clean_transport_display_vectors, clean_transport_magnitudes = normalize_vectors(
            clean_transport_vectors[clean_transport_indices].detach().cpu(),
            0.22,
        )
        noisy_grid_model_score_display, noisy_grid_model_score_magnitudes = (
            normalize_vectors(
                noisy_grid_model_score.detach().cpu(),
                noisy_vector_display_length,
            )
        )
        noisy_grid_gaussian_score_display, noisy_grid_gaussian_score_magnitudes = (
            normalize_vectors(
                noisy_grid_gaussian_score.detach().cpu(),
                noisy_vector_display_length,
            )
        )
        clean_grid_score_difference_display, clean_grid_score_difference_magnitudes = (
            normalize_vectors(
                clean_grid_score_difference.detach().cpu(),
                clean_vector_display_length,
            )
        )
        clean_grid_transport_display, clean_grid_transport_magnitudes = (
            normalize_vectors(
                (-clean_grid_score_difference).detach().cpu(),
                clean_vector_display_length,
            )
        )
        snapshots.append(
            {
                "noise_level": noise_level,
                "noisy_latents": noisy_latents.detach().cpu(),
                "score_points": noisy_latents[score_indices].detach().cpu(),
                "score_vectors": score_display_vectors,
                "score_magnitudes": score_magnitudes,
                "gaussian_score_vectors": gaussian_score_display_vectors,
                "gaussian_score_magnitudes": gaussian_score_magnitudes,
                "noisy_drift_points": noisy_drift_points[noisy_drift_indices]
                .detach()
                .cpu(),
                "noisy_drift_vectors": noisy_drift_display_vectors,
                "noisy_drift_magnitudes": noisy_drift_magnitudes,
                "clean_latents": clean_latents.detach().cpu(),
                "clean_drift_points": clean_latents[clean_drift_indices].detach().cpu(),
                "clean_drift_vectors": clean_drift_display_vectors,
                "clean_drift_magnitudes": clean_drift_magnitudes,
                "clean_transport_points": clean_latents[clean_transport_indices]
                .detach()
                .cpu(),
                "clean_transport_vectors": clean_transport_display_vectors,
                "clean_transport_magnitudes": clean_transport_magnitudes,
                "noisy_contour_xs": noisy_xs,
                "noisy_contour_ys": noisy_ys,
                "noisy_mismatch_norm": noisy_mismatch_norm.detach().cpu(),
                "clean_contour_xs": clean_xs,
                "clean_contour_ys": clean_ys,
                "clean_pullback_mismatch_norm": clean_pullback_mismatch_norm.detach().cpu(),
                "clean_transport_potential": clean_transport_potential.detach().cpu(),
                "noisy_grid_score_points": noisy_vector_grid_points.detach().cpu(),
                "noisy_grid_score_vectors": noisy_grid_model_score_display,
                "noisy_grid_score_magnitudes": noisy_grid_model_score_magnitudes,
                "noisy_grid_gaussian_score_vectors": noisy_grid_gaussian_score_display,
                "noisy_grid_gaussian_score_magnitudes": noisy_grid_gaussian_score_magnitudes,
                "clean_grid_difference_points": clean_vector_grid_points.detach().cpu(),
                "clean_grid_difference_vectors": clean_grid_score_difference_display,
                "clean_grid_difference_magnitudes": clean_grid_score_difference_magnitudes,
                "clean_grid_transport_vectors": clean_grid_transport_display,
                "clean_grid_transport_magnitudes": clean_grid_transport_magnitudes,
                "mode_ids": mode_ids.detach().cpu(),
                "score_mode_ids": mode_ids[score_indices].detach().cpu(),
                "noisy_drift_mode_ids": mode_ids[noisy_drift_indices].detach().cpu(),
                "clean_drift_mode_ids": mode_ids[clean_drift_indices].detach().cpu(),
                "clean_transport_mode_ids": mode_ids[clean_transport_indices]
                .detach()
                .cpu(),
            }
        )

    if model_was_training:
        model.train()

    fig = make_subplots(
        rows=2,
        cols=4,
        subplot_titles=[
            "noise-critic score estimate",
            "noisy-latent descent for denoiser mismatch",
            "clean-latent pullback of mismatch descent",
            "noisy score mismatch norm + model / Gaussian score fields",
            "clean-latent descent field from pullback score difference",
            "clean pullback linear potential + descent field",
            "noise prediction loss sweep",
            "clean pullback mismatch norm + score difference field",
        ],
        horizontal_spacing=0.05,
        vertical_spacing=0.12,
    )

    toggle_trace_indices = []
    for snapshot_index, snapshot in enumerate(snapshots):
        visible = snapshot_index == 0
        start_trace = len(fig.data)
        for mode_id, color in enumerate(MODE_COLORS):
            point_mask = snapshot["mode_ids"] == mode_id
            score_mask = snapshot["score_mode_ids"] == mode_id
            noisy_drift_mask = snapshot["noisy_drift_mode_ids"] == mode_id
            clean_drift_mask = snapshot["clean_drift_mode_ids"] == mode_id
            clean_transport_mask = snapshot["clean_transport_mode_ids"] == mode_id
            fig.add_trace(
                go.Scatter(
                    x=snapshot["noisy_latents"][point_mask, 0].numpy(),
                    y=snapshot["noisy_latents"][point_mask, 1].numpy(),
                    mode="markers",
                    marker={"size": 4, "color": color, "opacity": 0.22},
                    name=f"noisy latent mode {mode_id}",
                    visible=visible,
                    showlegend=visible,
                ),
                row=1,
                col=1,
            )
            if int(score_mask.sum().item()) > 0:
                add_quiver_traces(
                    fig,
                    points=snapshot["score_points"][score_mask],
                    vectors=snapshot["score_vectors"][score_mask],
                    magnitudes=snapshot["score_magnitudes"][score_mask],
                    color=color,
                    name=f"score mode {mode_id}",
                    row=1,
                    col=1,
                    visible=visible,
                    showlegend=visible,
                )
                add_quiver_traces(
                    fig,
                    points=snapshot["score_points"][score_mask],
                    vectors=snapshot["gaussian_score_vectors"][score_mask],
                    magnitudes=snapshot["gaussian_score_magnitudes"][score_mask],
                    color="#2ca02c",
                    name="analytic Gaussian score",
                    row=1,
                    col=1,
                    visible=visible,
                    showlegend=visible and mode_id == 0,
                    arrow_scale=0.28,
                    line_width=1.1,
                    opacity=0.7,
                )
            fig.add_trace(
                go.Scatter(
                    x=snapshot["noisy_latents"][point_mask, 0].numpy(),
                    y=snapshot["noisy_latents"][point_mask, 1].numpy(),
                    mode="markers",
                    marker={"size": 4, "color": color, "opacity": 0.22},
                    name=f"noisy mismatch mode {mode_id}",
                    visible=visible,
                    showlegend=False,
                ),
                row=1,
                col=2,
            )
            if int(noisy_drift_mask.sum().item()) > 0:
                add_quiver_traces(
                    fig,
                    points=snapshot["noisy_drift_points"][noisy_drift_mask],
                    vectors=snapshot["noisy_drift_vectors"][noisy_drift_mask],
                    magnitudes=snapshot["noisy_drift_magnitudes"][noisy_drift_mask],
                    color=color,
                    name=f"noisy drift mode {mode_id}",
                    row=1,
                    col=2,
                    visible=visible,
                    showlegend=False,
                )
            fig.add_trace(
                go.Scatter(
                    x=snapshot["clean_latents"][point_mask, 0].numpy(),
                    y=snapshot["clean_latents"][point_mask, 1].numpy(),
                    mode="markers",
                    marker={"size": 4, "color": color, "opacity": 0.22},
                    name=f"clean latent mode {mode_id}",
                    visible=visible,
                    showlegend=False,
                ),
                row=1,
                col=3,
            )
            if int(clean_drift_mask.sum().item()) > 0:
                add_quiver_traces(
                    fig,
                    points=snapshot["clean_drift_points"][clean_drift_mask],
                    vectors=snapshot["clean_drift_vectors"][clean_drift_mask],
                    magnitudes=snapshot["clean_drift_magnitudes"][clean_drift_mask],
                    color=color,
                    name=f"clean drift mode {mode_id}",
                    row=1,
                    col=3,
                    visible=visible,
                    showlegend=False,
                )
            fig.add_trace(
                go.Scatter(
                    x=snapshot["clean_latents"][point_mask, 0].numpy(),
                    y=snapshot["clean_latents"][point_mask, 1].numpy(),
                    mode="markers",
                    marker={"size": 4, "color": color, "opacity": 0.22},
                    name=f"clean transport mode {mode_id}",
                    visible=visible,
                    showlegend=False,
                ),
                row=2,
                col=1,
            )
            if int(clean_transport_mask.sum().item()) > 0:
                add_quiver_traces(
                    fig,
                    points=snapshot["clean_transport_points"][clean_transport_mask],
                    vectors=snapshot["clean_transport_vectors"][clean_transport_mask],
                    magnitudes=snapshot["clean_transport_magnitudes"][
                        clean_transport_mask
                    ],
                    color=color,
                    name=f"clean transport mode {mode_id}",
                    row=2,
                    col=1,
                    visible=visible,
                    showlegend=False,
                )
            fig.add_trace(
                go.Scatter(
                    x=snapshot["noisy_latents"][point_mask, 0].numpy(),
                    y=snapshot["noisy_latents"][point_mask, 1].numpy(),
                    mode="markers",
                    marker={"size": 3, "color": color, "opacity": 0.16},
                    name=f"noisy contour support mode {mode_id}",
                    visible=visible,
                    showlegend=False,
                ),
                row=1,
                col=4,
            )
            fig.add_trace(
                go.Scatter(
                    x=snapshot["clean_latents"][point_mask, 0].numpy(),
                    y=snapshot["clean_latents"][point_mask, 1].numpy(),
                    mode="markers",
                    marker={"size": 3, "color": color, "opacity": 0.16},
                    name=f"clean contour support mode {mode_id}",
                    visible=visible,
                    showlegend=False,
                ),
                row=2,
                col=4,
            )
            fig.add_trace(
                go.Scatter(
                    x=snapshot["clean_latents"][point_mask, 0].numpy(),
                    y=snapshot["clean_latents"][point_mask, 1].numpy(),
                    mode="markers",
                    marker={"size": 3, "color": color, "opacity": 0.16},
                    name=f"clean transport support mode {mode_id}",
                    visible=visible,
                    showlegend=False,
                ),
                row=2,
                col=2,
            )

        add_mismatch_contour_trace(
            fig,
            xs=snapshot["noisy_contour_xs"],
            ys=snapshot["noisy_contour_ys"],
            mismatch_norm=snapshot["noisy_mismatch_norm"],
            row=1,
            col=4,
            visible=visible,
            name="noisy score mismatch norm",
        )
        add_mismatch_contour_trace(
            fig,
            xs=snapshot["clean_contour_xs"],
            ys=snapshot["clean_contour_ys"],
            mismatch_norm=snapshot["clean_pullback_mismatch_norm"],
            row=2,
            col=4,
            visible=visible,
            name="clean pullback mismatch norm",
        )
        add_quiver_traces(
            fig,
            points=snapshot["noisy_grid_score_points"],
            vectors=snapshot["noisy_grid_score_vectors"],
            magnitudes=snapshot["noisy_grid_score_magnitudes"],
            color="#111111",
            name="model score field",
            row=1,
            col=4,
            visible=visible,
            showlegend=snapshot_index == 0,
            show_hover=False,
            arrow_scale=0.42,
            line_width=1.7,
            opacity=0.9,
        )
        add_quiver_traces(
            fig,
            points=snapshot["noisy_grid_score_points"],
            vectors=snapshot["noisy_grid_gaussian_score_vectors"],
            magnitudes=snapshot["noisy_grid_gaussian_score_magnitudes"],
            color="#1f77b4",
            name="analytic Gaussian score field",
            row=1,
            col=4,
            visible=visible,
            showlegend=snapshot_index == 0,
            show_hover=False,
            arrow_scale=0.42,
            line_width=1.7,
            opacity=0.9,
        )
        add_model_and_gaussian_score_hover_trace(
            fig,
            points=snapshot["noisy_grid_score_points"],
            model_score_magnitudes=snapshot["noisy_grid_score_magnitudes"],
            gaussian_score_magnitudes=snapshot["noisy_grid_gaussian_score_magnitudes"],
            row=1,
            col=4,
            visible=visible,
        )
        add_quiver_traces(
            fig,
            points=snapshot["clean_grid_difference_points"],
            vectors=snapshot["clean_grid_difference_vectors"],
            magnitudes=snapshot["clean_grid_difference_magnitudes"],
            color="#d62728",
            name="score difference field",
            row=2,
            col=4,
            visible=visible,
            showlegend=snapshot_index == 0,
            arrow_scale=0.42,
            line_width=1.7,
            opacity=0.9,
        )
        add_potential_contour_trace(
            fig,
            xs=snapshot["clean_contour_xs"],
            ys=snapshot["clean_contour_ys"],
            potential=snapshot["clean_transport_potential"],
            row=2,
            col=2,
            visible=visible,
            name="clean pullback linear potential",
        )
        add_quiver_traces(
            fig,
            points=snapshot["clean_grid_difference_points"],
            vectors=snapshot["clean_grid_transport_vectors"],
            magnitudes=snapshot["clean_grid_transport_magnitudes"],
            color="#2ca02c",
            name="descent field",
            row=2,
            col=2,
            visible=visible,
            showlegend=snapshot_index == 0,
            arrow_scale=0.42,
            line_width=1.7,
            opacity=0.9,
        )
        toggle_trace_indices.append(list(range(start_trace, len(fig.data))))

    contour_trace_start = len(fig.data)
    for row, cols in [(1, [1, 2, 3, 4]), (2, [1, 2, 4])]:
        for col in cols:
            add_latent_contour_traces(
                fig,
                contour_points=contour_points,
                contour_ids=contour_ids,
                contour_levels=contour_levels,
                row=row,
                col=col,
                showlegend=row == 1 and col == 1,
            )
    contour_trace_indices = list(range(contour_trace_start, len(fig.data)))

    fig.add_trace(
        go.Scatter(
            x=plot_t_values,
            y=loss_estimates,
            mode="lines+markers",
            line={"width": 3, "color": "#d62728"},
            marker={"size": 8, "color": "#d62728"},
            name="noise prediction loss",
        ),
        row=2,
        col=3,
    )

    line_trace_index = len(fig.data) - 1
    buttons = []
    for snapshot_index, snapshot in enumerate(snapshots):
        visibility = [False] * len(fig.data)
        for trace_index in toggle_trace_indices[snapshot_index]:
            visibility[trace_index] = True
        for trace_index in contour_trace_indices:
            visibility[trace_index] = True
        visibility[line_trace_index] = True
        buttons.append(
            {
                "label": f"t={snapshot['noise_level']:.2f}",
                "method": "update",
                "args": [
                    {
                        "visible": visibility,
                    },
                    {
                        "title": (
                            "critic diagnostics at step "
                            f"{step} for t={snapshot['noise_level']:.2f}"
                        ),
                        "xaxis4.range": [
                            float(snapshot["noisy_contour_xs"][0].item()),
                            float(snapshot["noisy_contour_xs"][-1].item()),
                        ],
                        "yaxis4.range": [
                            float(snapshot["noisy_contour_ys"][0].item()),
                            float(snapshot["noisy_contour_ys"][-1].item()),
                        ],
                        "xaxis6.range": [
                            float(snapshot["clean_contour_xs"][0].item()),
                            float(snapshot["clean_contour_xs"][-1].item()),
                        ],
                        "yaxis6.range": [
                            float(snapshot["clean_contour_ys"][0].item()),
                            float(snapshot["clean_contour_ys"][-1].item()),
                        ],
                        "xaxis8.range": [
                            float(snapshot["clean_contour_xs"][0].item()),
                            float(snapshot["clean_contour_xs"][-1].item()),
                        ],
                        "yaxis8.range": [
                            float(snapshot["clean_contour_ys"][0].item()),
                            float(snapshot["clean_contour_ys"][-1].item()),
                        ],
                    },
                ],
            }
        )

    for row, col, y_axis in [
        (1, 1, "y"),
        (1, 2, "y2"),
        (1, 3, "y3"),
        (1, 4, "y4"),
        (2, 1, "y5"),
        (2, 2, "y6"),
        (2, 4, "y8"),
    ]:
        fig.update_xaxes(
            showgrid=True,
            zeroline=False,
            scaleanchor=y_axis,
            scaleratio=1,
            row=row,
            col=col,
        )
        fig.update_yaxes(
            showgrid=True,
            zeroline=False,
            row=row,
            col=col,
        )
    fig.update_xaxes(
        range=[
            float(snapshots[0]["noisy_contour_xs"][0].item()),
            float(snapshots[0]["noisy_contour_xs"][-1].item()),
        ],
        row=1,
        col=4,
    )
    fig.update_yaxes(
        range=[
            float(snapshots[0]["noisy_contour_ys"][0].item()),
            float(snapshots[0]["noisy_contour_ys"][-1].item()),
        ],
        row=1,
        col=4,
    )
    fig.update_xaxes(
        range=[
            float(snapshots[0]["clean_contour_xs"][0].item()),
            float(snapshots[0]["clean_contour_xs"][-1].item()),
        ],
        row=2,
        col=2,
    )
    fig.update_yaxes(
        range=[
            float(snapshots[0]["clean_contour_ys"][0].item()),
            float(snapshots[0]["clean_contour_ys"][-1].item()),
        ],
        row=2,
        col=2,
    )
    fig.update_xaxes(
        range=[
            float(snapshots[0]["clean_contour_xs"][0].item()),
            float(snapshots[0]["clean_contour_xs"][-1].item()),
        ],
        row=2,
        col=4,
    )
    fig.update_yaxes(
        range=[
            float(snapshots[0]["clean_contour_ys"][0].item()),
            float(snapshots[0]["clean_contour_ys"][-1].item()),
        ],
        row=2,
        col=4,
    )
    fig.update_xaxes(
        title_text="noise level t",
        tickmode="array",
        tickvals=plot_t_values,
        showgrid=True,
        zeroline=False,
        row=2,
        col=3,
    )
    fig.update_yaxes(
        title_text="E[||eps_theta(y_t, t) - eps||^2]",
        showgrid=True,
        zeroline=False,
        row=2,
        col=3,
    )
    fig.update_layout(
        title=f"critic diagnostics at step {step} for t={plot_t_values[0]:.2f}",
        height=880,
        width=2700,
        margin={"l": 30, "r": 20, "t": 90, "b": 35},
        legend={"x": 1.02, "xanchor": "left", "y": 1.0, "yanchor": "top"},
        updatemenus=[
            {
                "type": "dropdown",
                "direction": "down",
                "buttons": buttons,
                "x": 0.0,
                "xanchor": "left",
                "y": 1.11,
                "yanchor": "top",
                "showactive": True,
            }
        ],
    )
    return fig


def snapshot_progress(
    scatter_num_samples: int = 1024,
    vector_num_samples: int = 64,
    plot_t_values: list[float] = [
        0.05,
        # 0.1,
        # 0.2,
        # 0.3,
        # 0.4,
        # 0.5,
        # 0.6,
        0.7,
        # 0.8,
        # 0.9,
        # 0.95,
    ],
    show: bool = True,
) -> dict[str, go.Figure]:
    figures = {
        "losses": plot_training_losses(),
        "contours_and_reconstructions": plot_model_contours_and_reconstructions(
            scatter_num_samples=scatter_num_samples,
        ),
        "critic_diagnostics": plot_score_estimates(
            scatter_num_samples=scatter_num_samples,
            vector_num_samples=vector_num_samples,
            plot_t_values=plot_t_values,
        ),
    }
    if show:
        for figure in figures.values():
            figure.show()
    return figures



def clamp_vector_norms(
    vectors: Float[Tensor, "batch 2"],
    max_norm: float,
) -> Float[Tensor, "batch 2"]:
    norms = vectors.norm(dim=1, keepdim=True).clamp_min(1e-6)
    return vectors * torch.clamp(max_norm / norms, max=1.0)


def explicit_chart_drift_targets(
    clean_latents: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
    noise: Float[Tensor, "batch 2"],
    drift_step_size: float,
    max_transport_norm: float,
) -> tuple[
    Float[Tensor, "batch 2"],
    Float[Tensor, "batch 2"],
    Float[Tensor, "batch 2"],
]:
    with torch.no_grad():
        noised_latents = (1.0 - t) * clean_latents + t * noise
        clean_pullback_score_difference = (1.0 - t) * evaluate_score_difference(
            noised_latents,
            t=t,
        )
        clean_transport = clamp_vector_norms(
            -clean_pullback_score_difference,
            max_norm=max_transport_norm,
        )
        drifted_latents = clean_latents + drift_step_size * clean_transport
    return drifted_latents, clean_transport, noised_latents


In [ ]:
start_denoising_after = 100
start_sparse_chart_updates_after = 1500
chart_update_every = 3
chart_drift_step_size = 0.5
max_chart_transport_norm = 5.0

for step in tqdm(range(1, 150000)):
    encoder_op.zero_grad(set_to_none=True)
    decoder_op.zero_grad(set_to_none=True)
    critic_op.zero_grad(set_to_none=True)

    x = data_config.sample(batch_size).to(device)
    prior = data_config.sample_prior(batch_size).to(device)

    with torch.no_grad():
        critic_clean_latent = model.encoder(x)
    critic_t = torch.rand(size=(batch_size, 1), device=device) * 0.975 + 0.025
    critic_noise = torch.randn((batch_size, 2), device=device)
    noised_data_latent = (1.0 - critic_t) * critic_clean_latent + critic_t * critic_noise
    noise_preds = model.noise_critic(noised_data_latent, t=critic_t)
    critic_denoising_loss = (
        F.huber_loss(noise_preds, critic_noise, delta=5.0, reduction="none")
        .sum(-1)
        .mean()
    )
    if step < start_denoising_after:
        critic_denoising_loss = critic_denoising_loss * 0.0
    critic_denoising_loss.backward()
    torch.nn.utils.clip_grad_norm_(
        model.noise_critic.parameters(),
        max_norm=grad_clip_norm,
    )
    critic_op.step()

    apply_chart_update = step < start_sparse_chart_updates_after or (
        (step - start_sparse_chart_updates_after) % chart_update_every == 0
    )
    apply_chart_drift = (
        step >= start_sparse_chart_updates_after and apply_chart_update
    )

    data_cycle_loss = x.new_tensor(0.0)
    prior_cycle_loss = x.new_tensor(0.0)
    encoder_chart_drift_loss = x.new_tensor(0.0)
    decoder_chart_drift_loss = x.new_tensor(0.0)
    prior_loss = x.new_tensor(0.0)

    if apply_chart_update:
        encoder_op.zero_grad(set_to_none=True)
        decoder_op.zero_grad(set_to_none=True)

        data_latent = model.encoder(x)
        cycle_data = model.decoder(data_latent)
        model_samples = model.decoder(prior)
        cycle_prior = model.encoder(model_samples)

        data_cycle_loss = F.huber_loss(cycle_data, x, delta=3.0)
        prior_cycle_loss = F.huber_loss(cycle_prior, prior, delta=3.0)
        prior_loss = (
            (data_latent.mean() ** 2)
            + 0.001 * data_latent.pow(2).mean()
        ) * 1e-4

        if apply_chart_drift:
            chart_t = torch.rand(size=(batch_size, 1), device=device) * 0.975 + 0.025
            chart_noise = torch.randn((batch_size, 2), device=device)
            drifted_latent_targets, _, _ = explicit_chart_drift_targets(
                clean_latents=data_latent.detach(),
                t=chart_t,
                noise=chart_noise,
                drift_step_size=chart_drift_step_size,
                max_transport_norm=max_chart_transport_norm,
            )
            encoder_chart_drift_loss = F.huber_loss(
                data_latent,
                drifted_latent_targets,
                delta=1.0,
            )
            decoder_chart_drift_loss = F.huber_loss(
                model.decoder(drifted_latent_targets),
                x,
                delta=3.0,
            )

        total_chart_loss = (
            data_cycle_loss
            + prior_cycle_loss
            + prior_loss
            + encoder_chart_drift_loss
            + decoder_chart_drift_loss
        )
        total_chart_loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.encoder.parameters(),
            max_norm=grad_clip_norm,
        )
        torch.nn.utils.clip_grad_norm_(
            model.decoder.parameters(),
            max_norm=grad_clip_norm,
        )
        encoder_op.step()
        decoder_op.step()
        metric_latent = data_latent.detach()
    else:
        with torch.no_grad():
            metric_latent = model.encoder(x)

    latent_mean = metric_latent.mean(dim=0)
    centered_latent = metric_latent - latent_mean
    latent_cov = centered_latent.T @ centered_latent / max(
        centered_latent.shape[0] - 1,
        1,
    )
    latent_mean_signal = latent_mean.square().sum()
    latent_cov_signal = (
        latent_cov
        - torch.eye(latent_cov.shape[0], device=latent_cov.device, dtype=latent_cov.dtype)
    ).square().sum()

    logs["losses"]["data_cycle"].append(data_cycle_loss.item())
    logs["losses"]["prior_cycle"].append(prior_cycle_loss.item())
    logs["losses"]["critic_denoising"].append(critic_denoising_loss.item())
    logs["losses"]["encoder_chart_drift"].append(encoder_chart_drift_loss.item())
    logs["losses"]["decoder_chart_drift"].append(decoder_chart_drift_loss.item())
    logs["latent_metrics"]["mean_signal"].append(latent_mean_signal.item())
    logs["latent_metrics"]["cov_signal"].append(latent_cov_signal.item())

    if step in [1] or step % 1000 == 0:
        snapshot_progress()